# Import Modules

In [ ]:
#TO PREPROCESS THE DATA 
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.io import loadmat
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
from math import log, sqrt
from sklearn.metrics import confusion_matrix
import seaborn as sns
from sklearn.utils import shuffle
import time
from sklearnex import patch_sklearn
from sklearn.model_selection import ShuffleSplit

In [ ]:
patch_sklearn()

# Extract the data

In [ ]:
X=loadmat("data/PaviaU.mat")['paviaU']
Y=loadmat("data/PaviaU_gt.mat")['paviaU_gt']

X.shape,Y.shape

# Visualize the image

In [ ]:
def plot_random_bands(X, num_subplots=8, figsize=(18, 10), cmap='Set3'):
    sns.set_style('darkgrid')
    fig = plt.figure(figsize=figsize)
    
    band_indices = np.random.choice(X.shape[2], size=num_subplots, replace=False)

    for i, ran_val in enumerate(band_indices, start=1):
        fig.add_subplot(2, 4, i)
        plt.imshow(X[:, :, ran_val], cmap=cmap)
        plt.axis("off")
        plt.title(f'band-{ran_val}')

In [ ]:
plot_random_bands(X)

# Visualize Ground Truth

In [ ]:
def plot_Ground_Truth(Y, figsize=(10, 8), cmap='Set3'):
    plt.figure(figsize=figsize)
    plt.imshow(Y, cmap=cmap)
    plt.colorbar()
    plt.axis('off')
    plt.show()

In [ ]:
plot_Ground_Truth(Y)

# Display the number of each category

In [ ]:
def plot_band_boxplot(X,Y):
    plt.figure(figsize=(16, 6))
    df=pd.DataFrame(X.reshape(X.shape[0]*X.shape[1], -1))
    df.colmns= [i for i in range(1, df.shape[-1]+1)]
    df['class'] = Y.ravel()
    sns.boxplot(x=df["class"], y=df[0], width=0.3)
    plt.title('Box Plot', fontsize=16)
    plt.xlabel('Class', fontsize=14)
    plt.show()
    return df

In [ ]:
df=plot_band_boxplot(X,Y)

In [ ]:
df.columns = [f'band-{i}' for i in range(1, 1+X.shape[2])]+['class']
df

# Denoise the data set

In [ ]:
def Denoise_dataset(dataset):
    Denoised_x= df[df['class']!=0].iloc[:, :-1].values
    Denoised_y = df[df['class']!=0].iloc[:, -1].values
    Denoised_df = pd.concat([pd.DataFrame(Denoised_x), pd.Series(Denoised_y)], axis=1)
    Denoised_df.columns = [f'band-{i}' for i in range(1, 1+X.shape[2])]+['class']
    return Denoised_df

In [ ]:
Denoised_df=Denoise_dataset(df)

# Feature-Selection with Sequential backward Search

In [ ]:
import statsmodels.api as sm
def backward_elimination(data, target,significance_level = 0.05):
    features = data.columns.tolist()
    while(len(features)>0):
        features_with_constant = sm.add_constant(data[features])
        p_values = sm.OLS(target, features_with_constant).fit().pvalues[1:]
        max_p_value = p_values.max()
        if(max_p_value >= significance_level):
            excluded_feature = p_values.idxmax()
            features.remove(excluded_feature)
        else:
            break 
    return features

forward_start_time = time.time()
X=Denoised_df.iloc[:, :-1]
y=Denoised_df.iloc[:, -1]
best_features=backward_elimination(X,y)
print('Time taken by backward Feature Selection is :'+str(int(time.time() - forward_start_time))+' seconds')

df= pd.concat([Denoised_df[best_features], y], axis=1)
df

In [ ]:
forward_start_time = time.time()
X=Denoised_df.iloc[:, :-1]
y=Denoised_df.iloc[:, -1]
best_features=backward_elimination(X,y)
print('Time taken by backward Feature Selection is :'+str(int(time.time() - forward_start_time))+' seconds')

In [ ]:
df= pd.concat([Denoised_df[best_features], y], axis=1)
df

# Split the data set

In [ ]:
def split_dataset(Denoised_df, test_size=0.2, random_state=42):
    unique_classes = set(Denoised_df.iloc[:,-1])
    train_data = {class_label: {'X': None, 'y': None} for class_label in unique_classes}
    test_data = {class_label: {'X': None, 'y': None} for class_label in unique_classes}
    for class_label in unique_classes:
        class_indices = [i for i, label in enumerate(Denoised_df.iloc[:,-1]) if label == class_label]
        X_class, y_class = Denoised_df.iloc[class_indices,:-1], Denoised_df.iloc[class_indices,-1]
        X_train, X_test, y_train, y_test = train_test_split(X_class, y_class, test_size=test_size, random_state=random_state)
        train_data[class_label]['X'] = X_train
        train_data[class_label]['y'] = y_train
        test_data[class_label]['X'] = X_test
        test_data[class_label]['y'] = y_test
        
    dfs_to_concat_X = [train_data[i]['X'] for i in unique_classes]
    dfs_to_concat_Y = [train_data[i]['y'] for i in unique_classes]
    merged_train_df_X = pd.concat(dfs_to_concat_X, ignore_index=True)
    merged_train_df_Y=  pd.concat(dfs_to_concat_Y, ignore_index=True)
    train_data = pd.concat([merged_train_df_X, merged_train_df_Y], axis=1)
    dfs_to_concat_X= [test_data[i]['X'] for i in unique_classes]
    dfs_to_concat_Y= [test_data[i]['y'] for i in unique_classes]
    merged_train_df_X = pd.concat(dfs_to_concat_X, ignore_index=True)
    merged_train_df_Y=  pd.concat(dfs_to_concat_Y, ignore_index=True)
    test_data= pd.concat([merged_train_df_X, merged_train_df_Y], axis=1)
    
    train_data=shuffle(train_data, random_state=42)
    test_data=shuffle(test_data, random_state=42)
    
    x_train= train_data.iloc[:, :-1]
    x_test=  test_data.iloc[:,:-1]
    y_train= train_data.iloc[:,-1]
    y_test= test_data.iloc[:,-1]    
    return x_train,x_test,y_train,y_test

In [ ]:
x_train,x_test,y_train,y_test=split_dataset(df)

In [ ]:
x_train.shape,x_test.shape,y_train.shape,y_test.shape

In [ ]:
class PrincipalComponentAnalysis:
    def __init__(self, n_components):
        self.n_components = n_components
        self.components = None
        self.mean = None
        self.explained_variance_ratio_ = None #New parameters 
    
    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        X = X - self.mean
        cov = np.cov(X.T)
        eigenvalues, eigenvectors = np.linalg.eig(cov)
        eigenvectors = eigenvectors.T
        idxs = np.argsort(-eigenvalues)  
        eigenvalues = eigenvalues[idxs]
        eigenvectors = eigenvectors[idxs]
        self.components = eigenvectors[:self.n_components]
        total_variance = np.sum(eigenvalues)
        explained_variance = eigenvalues[:self.n_components]
        self.explained_variance_ratio_ = explained_variance / total_variance
    
    def transform(self, X):
        X = X - self.mean
        return np.dot(X, self.components.T)
    # New function
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

# Implement Random Forest

In [ ]:
def unique_counts(labels):
    """
    Unique count function used to count labels.
    """
    results = {}
    for label in labels:
        value = label.item()
        if value not in results.keys():
            results[value] = 0
        results[value] += 1
    return results


def divide_set(vectors, labels, column, value):
    """
    Divide the sets into two different sets along a specific dimension and value.
    """
    set_1 = [(vector, label) for vector, label in zip(vectors, labels) if split_function(vector, column, value)]
    set_2 = [(vector, label) for vector, label in zip(vectors, labels) if not split_function(vector, column, value)]
    vectors_set_1 = [element[0] for element in set_1]
    vectors_set_2 = [element[0] for element in set_2]
    label_set_1 = [element[1] for element in set_1]
    label_set_2 = [element[1] for element in set_2]

    return vectors_set_1, label_set_1, vectors_set_2, label_set_2


def split_function(vector, column, value):
    """
    Split function
    """
    return vector[column] >= value


def log2(x):
    """
    Log2 function
    """
    return log(x) / log(2)


def sample_vectors(vectors, labels, nb_samples):
    """
    Sample vectors and labels uniformly.
    """
    sampled_indices = torch.LongTensor(random.sample(range(len(vectors)), nb_samples))
    sampled_vectors = torch.index_select(vectors,0, sampled_indices)
    sampled_labels = torch.index_select(labels,0, sampled_indices)

    return sampled_vectors, sampled_labels


def sample_dimensions(vectors):
    """
    Sample vectors along dimension uniformly.
    """
    sample_dimension = torch.LongTensor(random.sample(range(len(vectors[0])), int(sqrt(len(vectors[0])))))

    return sample_dimension


def entropy(labels):
    """
    Entropy function.
    """
    results = unique_counts(labels)
    ent = 0.0
    for r in results.keys():
        p = float(results[r]) / len(labels)
        ent = ent - p * log2(p)
    return ent


def variance(values):
    """
    Variance function.
    """
    mean_value = mean(values)
    var = 0.0
    for value in values:
        var = var + torch.sum(torch.sqrt(torch.pow(value-mean_value,2))).item()/len(values)
    return var


def mean(values):
    """
    Mean function.
    """
    m = 0.0
    for value in values:
        m = m + value/len(values)
    return m

def gini(labels):
    total = len(labels)
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
    impurity = 1
    for label in counts:
        p1 = counts[label] / total
        impurity -= p1**2
    return impurity


# Build Tree nodes

In [ ]:
class DecisionNode:
    def __init__(self, col=-1, value=None, results=None, tb=None, fb=None):
        self.col = col
        self.value = value
        self.results = results
        self.tb = tb
        self.fb = fb

In [ ]:
import torch
from math import log2
class TorchDecisionTreeClassifier(torch.nn.Module):
    def __init__(self, max_depth=-1):
        super(TorchDecisionTreeClassifier, self).__init__()
        self._root_node = None
        self.max_depth = max_depth

    def fit(self, vectors, labels, criterion=None):
        if criterion is None:
            criterion = 'entropy'
        if criterion not in ['entropy', 'gini']:
            raise ValueError("Criterion must be either 'entropy' or 'gini'")

        func = entropy if criterion == 'entropy' else gini
        self._root_node = self._build_tree(vectors, labels, func, self.max_depth)

    def _build_tree(self, vectors, labels, func, depth):
        if len(vectors) == 0:
            return DecisionNode()
        if depth == 0:
            return DecisionNode(results=unique_counts(labels))

        best_gain_ratio, best_criteria, best_sets = self.find_best_gain_ratio(vectors, labels, func)

        if best_gain_ratio > 0:
            true_branch = self._build_tree(best_sets[0][0], best_sets[0][1], func, depth - 1)
            false_branch = self._build_tree(best_sets[1][0], best_sets[1][1], func, depth - 1)
            return DecisionNode(col=best_criteria[0], value=best_criteria[1], tb=true_branch, fb=false_branch)
        else:
            return DecisionNode(results=unique_counts(labels))

    def find_best_gain_ratio(self, vectors, labels, func):
        current_entropy = func(labels)
        best_gain_ratio = 0.0
        best_criteria = None
        best_sets = None

        column_count = len(vectors[0])
        for col in range(0, column_count):
            column_values = self.get_column_values(vectors, col)
            for value in column_values.keys():
                vectors_set_1, label_set_1, vectors_set_2, label_set_2 = divide_set(vectors, labels, col, value)

                p = float(len(vectors_set_1)) / len(vectors)
                if p == 1:
                    continue

                info_gain = current_entropy - p * func(label_set_1) - (1 - p) * func(label_set_2)
                split_info = -1 * (p * log2(p) + (1 - p) * log2(1 - p))
                gain_ratio = info_gain / split_info

                if gain_ratio > best_gain_ratio and len(vectors_set_1) > 0 and len(vectors_set_2) > 0:
                    best_gain_ratio = gain_ratio
                    best_criteria = (col, value)
                    best_sets = ((vectors_set_1, label_set_1), (vectors_set_2, label_set_2))

        return best_gain_ratio, best_criteria, best_sets

    def get_column_values(self, vectors, col):
        column_values = {}
        for vector in vectors:
            column_values[vector[col]] = 1
        return column_values

    def predict(self, vector):
        return self._classify(vector, self._root_node)

    def _classify(self, vector, node):
        if node.results is not None:
            return list(node.results.keys())[0]
        else:
            branch = node.tb if split_function(vector, node.col, node.value) else node.fb
            return self._classify(vector, branch)

In [ ]:
class TorchRandomForestClassifier(torch.nn.Module):
    def __init__(self, nb_trees, nb_samples, max_depth=-1, bootstrap=True):
        super(TorchRandomForestClassifier, self).__init__()
        self.trees = []
        self.trees_features = []
        self.nb_trees = nb_trees
        self.nb_samples = nb_samples
        self.max_depth = max_depth
        self.bootstrap = bootstrap

    def fit(self, vectors, labels, criterion='entropy'):
        if criterion not in ['entropy', 'gini']:
            raise ValueError("Criterion must be either 'entropy' or 'gini'")

        for _ in range(self.nb_trees):
            tree = TorchDecisionTreeClassifier(self.max_depth)
            list_features = sample_dimensions(vectors)
            self.trees_features.append(list_features)

            if self.bootstrap:
                sampled_vectors, sample_labels = sample_vectors(vectors, labels, self.nb_samples)
                sampled_featured_vectors = torch.index_select(sampled_vectors, 1, list_features)
                tree.fit(sampled_featured_vectors, sample_labels, criterion)
            else:
                sampled_featured_vectors = torch.index_select(vectors, 1, list_features)
                tree.fit(sampled_featured_vectors, labels, criterion)

            self.trees.append(tree)

    def predict(self, dataset):
        predict = []
        for i in range(len(dataset)):
            predict_value = self.predict_vector(dataset[i, :])
            predict.append(predict_value)
        return torch.LongTensor(predict)

    def predict_vector(self, vector):
        predictions = []
        for tree, index_features in zip(self.trees, self.trees_features):
            sampled_vector = torch.index_select(vector, 0, index_features)
            predictions.append(tree.predict(sampled_vector))
        return max(set(predictions), key=predictions.count)

    def predict_score(self, y_true, y_pred):
        correct = 0
        total = len(y_true)
        for t, p in zip(y_true, y_pred):
            if t == p:
                correct += 1
        return correct / total

    def plot_confusion_matrix(self, y_true, y_pred):
        plt.figure(figsize=(10, 7))
        classes = ['Asphalt', 'Meadows', 'Gravel', 'Trees', 'Painted metal sheets',
                   'Bare Soil', 'Bitumen', 'Self-Blocking Bricks', 'Shadows']
        mat = confusion_matrix(y_pred, y_true)
        df_cm = pd.DataFrame(mat, index=classes, columns=classes)
        sns.heatmap(df_cm, annot=True, fmt='d')
        plt.show()

    def calculate_metrics_per_class(self, actual_labels, predicted_labels, class_labels):
        report = []
        for label in class_labels:
            correct_label_mask = torch.eq(actual_labels, label)
            actual_count = torch.sum(correct_label_mask).item()
            predicted_label_mask = torch.eq(predicted_labels, label)
            predicted_count = torch.sum(predicted_label_mask).item()
            true_positive = torch.sum(correct_label_mask & predicted_label_mask).item()
            precision = true_positive / max(predicted_count, 1)
            recall = true_positive / max(actual_count, 1)
            f1_score = 2 * (precision * recall) / max((precision + recall), 1)
            report.append([label, round(precision, 2), round(recall, 2), round(f1_score, 2), actual_count])
        return report

    def generate_classification_report(self, report):
        header = ["Class", "Precision", "Recall", "F1-Score", "Support"]
        table = [header] + report
        total_precision = sum(row[1] for row in report) / len(report)
        total_recall = sum(row[2] for row in report) / len(report)
        total_f1 = sum(row[3] for row in report) / len(report)
        total_support = sum(row[4] for row in report)
        macro_avg = ["macro avg", round(total_precision, 2), round(total_recall, 2), round(total_f1, 2), total_support]
        weighted_avg = ["weighted avg", round(total_precision, 2), round(total_recall, 2), round(total_f1, 2),
                        total_support]
        table.extend([macro_avg, weighted_avg])

        for row in table:
            print("{:<30} {:<10} {:<10} {:<10} {:<10}".format(*row))

    def get_params(self, deep=True):
        return {
            "nb_trees": self.nb_trees,
            "nb_samples": self.nb_samples,
            "max_depth": self.max_depth,
            "bootstrap": self.bootstrap
        }

    def set_params(self, **params):
        self.nb_trees = params.get("nb_trees", self.nb_trees)
        self.nb_samples = params.get("nb_samples", self.nb_samples)
        self.max_depth = params.get("max_depth", self.max_depth)
        self.bootstrap = params.get("bootstrap", self.bootstrap)
        return self

In [ ]:
x_train

In [ ]:
 def THE_BEST_SCORE_PCA(a, b, trainData, trainLabel, testData, testLabel,model):
        best_acc = -1  
        best_dim = None  
        best_y_pred = None  
        explained_variance = []
        accuracy=[]
        trainLabel= torch.LongTensor(trainLabel)
        testLabel=  torch.LongTensor(testLabel)
        
        for i in range(a, b + 1):
            print(f"Evaluating for {i} components")
            pca = PrincipalComponentAnalysis(i)
            tmp_x_train = pca.fit_transform(trainData)
            tmp_x_train = torch.FloatTensor(tmp_x_train.astype('float32'))
   
            
            model.fit(tmp_x_train, trainLabel)
            temp_x_test=pca.transform(testData)
            temp_x_test= torch.FloatTensor(temp_x_test.astype('float32'))
            
            y_pred = model.predict(temp_x_test)
            
            
            accuracy_score = model.predict_score(testLabel, y_pred)
            explained_variance.append(np.sum(pca.explained_variance_ratio_)) 
            accuracy.append(accuracy_score)
            
            if accuracy_score > best_acc:
                best_acc = accuracy_score
                best_dim = i
                best_y_pred = y_pred
                best_dataset= tmp_x_train
                best_testData=  temp_x_test
                                     
        print(f"The highest accuracy is {best_acc}, achieved with {best_dim} components.")
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        plt.plot(range(a, b + 1), accuracy, marker='o')
        plt.title('Model Accuracy vs. Number of Components')
        plt.xlabel('Number of Components')
        plt.ylabel('Accuracy')
        plt.grid()

        plt.subplot(1, 2, 2)
        plt.plot(range(a, b + 1), explained_variance, marker='o', color='orange')
        plt.title('Explained Variance vs. Number of Components')
        plt.xlabel('Number of Components')
        plt.ylabel('Explained Variance')
        plt.grid()

        plt.tight_layout()
        plt.show()
        return best_testData, best_dataset,best_y_pred

In [ ]:
import random

start_time = time.time()
RandomForest= TorchRandomForestClassifier(nb_trees=40, nb_samples=200, max_depth=10, bootstrap=True)
best_testData, best_dataset,best_y_pred= THE_BEST_SCORE_PCA(3,10, x_train.values, y_train.values, x_test.values, y_test.values,RandomForest)
end_time = time.time()
runtime = end_time - start_time
print(f"Code execution time: {runtime} seconds")

In [ ]:
y_test= torch.FloatTensor(y_test.values)
y_train= torch.FloatTensor(y_train.values.astype('float32'))

class_labels=np.unique(y_train)
report= RandomForest.calculate_metrics_per_class(y_test,best_y_pred,class_labels)
RandomForest.generate_classification_report(report)

In [ ]:
RandomForest.plot_confusion_matrix(y_test,best_y_pred)

# Perform ten sets of cross-validation

In [ ]:
def crossValidation(model,trainData,trainLabel):
        cv = ShuffleSplit(n_splits=10, test_size=0.1, random_state=20)
        scores = cross_val_score(model, trainData, trainLabel, cv=cv, scoring='accuracy')
        print(f"Accuracy: {scores.mean():.2f} ± {scores.std():.2f}")

        plt.figure(figsize=(10, 5))
        plt.plot(range(len(scores)), scores, marker='o', linestyle='-', color='blue', label='Individual cross-validation scores')
        plt.axhline(y=scores.mean(), color='r', linestyle='--', label=f'Mean Accuracy: {scores.mean():.2f}')
        plt.axhline(y=scores.mean() + scores.std(), color='g', linestyle='--', label=f'Mean Accuracy + 1 Std Dev: {scores.mean() + scores.std():.2f}')
        plt.axhline(y=scores.mean() - scores.std(), color='y', linestyle='--', label=f'Mean Accuracy - 1 Std Dev: {scores.mean() - scores.std():.2f}')
        plt.xlabel('Iteration')
        plt.ylabel('Accuracy')
        plt.title('Cross-Validation Scores')
        plt.legend()
        plt.show()


In [ ]:
from sklearn.model_selection import cross_val_score
start_time = time.time()
RandomForest= TorchRandomForestClassifier(nb_trees=40, nb_samples=200, max_depth=10, bootstrap=True)
scores = crossValidation(RandomForest, best_dataset,y_train)

end_time = time.time()
runtime = end_time - start_time
print(f"Code execution time: {runtime} seconds")

# Grid search using GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {
    'nb_trees': np.arange(40, 50, 5),  
    'max_depth': np.arange(10, 20, 5), 
    'nb_samples': np.arange(150,250,50),
    'bootstrap': [True]
}
gs = GridSearchCV(RandomForest, param_grid, scoring='accuracy', cv=3, verbose=2)

In [ ]:
start_time = time.time()
gs.fit(best_dataset,  y_train)
print("Best Parameters:", gs.best_params_)
print("Best Cross-Validation Score:", gs.best_score_)
end_time=  time.time()
runtime = end_time - start_time
print(f"Code execution time: {runtime} seconds")

# Plot ROC roc curve

In [ ]:
def plot_roc_curve(model,testData,testLabel):
        """
        绘制ROC曲线。
        """
        # 获取每个类别的概率得分
        y_score = model.predict_proba(testData)
        
        n_classes=len(np.unique(testData))
        # 二值化标签（one-hot encoding）
        y_test_bin = label_binarize(testLabel, classes=np.arange(1, n_classes+1))
        n_classes=9
    

        # 计算每个类别的ROC曲线和AUC
        fpr = dict()
        tpr = dict()
        roc_auc = dict()
        for i in range(n_classes):
            fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])

        # 绘制所有ROC曲线
        plt.figure(figsize=(10, 10))
        colors = cycle(['aqua', 'darkorange', 'cornflowerblue', 'green', 'red', 'purple', 'pink', 'yellow', 'grey'])
        for i, color in zip(range(n_classes), colors):
            plt.plot(fpr[i], tpr[i], color=color, lw=2,
                     label=f'ROC curve of class {i} (area = {roc_auc[i]:.2f})')

        plt.plot([0, 1], [0, 1], 'k--', lw=2)
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Receiver Operating Characteristic')
        plt.legend(loc="lower right")
        plt.show()
        


In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from itertools import cycle
start_time = time.time()

clf = RandomForestClassifier(
           n_estimators=45,
           criterion="entropy",
           max_depth= 10
)
x_train= np.array(best_dataset)
y_train=np.array(y_train)
x_test=np.array(best_testData)
y_test=np.array(y_test)


clf.fit(x_train, y_train)
plot_roc_curve(clf,x_test,y_test)

end_time = time.time()
runtime = end_time - start_time
print(f"Code execution time: {runtime} seconds")